# 08 · Desbalanceamento entre famílias de produtos

**Objetivo:** verificar se as conclusões obtidas de qualquer movimentação representam Banda Larga (internet fixa), TV, Fone e Móvel. **Fonte:** métricas agregadas calculadas sobre a amostra confidencial; sem linhas nem IDs neste notebook.

A mesma linha pode contribuir para vários produtos: são quatro desfechos binários não exclusivos. A presença da família em M0 define a população elegível.

## 1. Contexto e método

População: registros `RETIDO` com a família presente em M0. Evento: flag de movimentação da família. Treino: jun–set/2025; validação: out/2025; teste: nov–dez/2025, eliminando IDs compartilhados entre períodos. Comparamos prior, regressão logística natural e regressão com `class_weight=balanced`. O conjunto de teste preserva as prevalências originais.

**Perguntas:** (i) o ranking por volume muda ao usar exposição? (ii) a classe positiva é rara em cada família? (iii) a ponderação melhora a discriminação ou prejudica Brier/calibração?

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
root = Path.cwd()
if not (root / 'pyproject.toml').exists(): root = root.parent
study = json.loads((root / 'reports/family_imbalance_2025.json').read_text(encoding='utf-8'))
labels = {'BL':'Banda Larga (internet fixa)', 'TV':'TV', 'FONE':'Fone', 'MV':'Móvel'}
products = list(labels)
assert all(study['exposures'][p]['n'] >= study['exposures'][p]['positive'] > 0 for p in products)
print(study['metadata']['population'])

## 2. Volume e taxa dentro dos elegíveis

Contagens brutas privilegiam famílias mais frequentes na carteira. O denominador correto para substituições é a presença da família em M0.

In [ ]:
exposure = study['exposures']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(products, [exposure[p]['positive'] for p in products]);axes[0].set_title('Movimentações observadas');axes[0].set_ylabel('Registros')
axes[1].bar(products, [100*exposure[p]['rate'] for p in products]);axes[1].set_title('Movimentação / elegíveis em M0');axes[1].set_ylabel('%')
for ax in axes: ax.grid(axis='y', alpha=.16)
plt.tight_layout();plt.show()
for p in products:print(f"{labels[p]}: {exposure[p]['positive']:,}/{exposure[p]['n']:,} = {exposure[p]['rate']:.1%}")

## 3. Prevalência do evento dentro de cada família

A raridade da família no conjunto total e o desequilíbrio entre positivos e negativos dentro dela são conceitos diferentes. Resultados de AP devem ser interpretados contra sua prevalência.

In [ ]:
families=study['models']['families']
for p in products:
 r=families[p]
 print(f"{labels[p]} | n teste={r['split_n']['test']:,} | prevalência treino={r['prevalence']['train']:.1%} | teste={r['prevalence']['test']:.1%}")

## 4. Sensibilidade à ponderação de classes

Comparar três alternativas no mesmo teste *por família*: prevalência constante, logística natural e logística ponderada. AUC e AP medem ranking; Brier mede erro quadrático das probabilidades. A ponderação não corrige automaticamente seleção por elegibilidade e pode piorar calibração.

In [ ]:
names = ['prior','logistic_natural','logistic_balanced']
for p in products:
 for n in names:
  m = families[p]['model_comparisons'][n]['test']
  print(f"{labels[p]:30} {n:18} AP={m['average_precision']:.3f} AP/base={m['ap_lift']:.2f} AUC={m['auc']:.3f} Brier={m['brier']:.4f} P@10={m['precision_top10']:.1%} R@10={m['recall_top10']:.1%}")
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
x=np.arange(len(products))
for i,n in enumerate(names):
 xx=x+(i-1)*.26
 axs[0].bar(xx,[families[p]['model_comparisons'][n]['test']['average_precision'] for p in products],width=.25,label=n)
 axs[1].bar(xx,[families[p]['model_comparisons'][n]['test']['brier'] for p in products],width=.25,label=n)
for ax,title in zip(axs,['Average precision no teste','Brier no teste (menor é melhor)']):
 ax.set_xticks(x,products); ax.set_title(title); ax.grid(axis='y',alpha=.15)
axs[0].legend(fontsize=8);plt.tight_layout();plt.show()

## 5. Resposta de negócio e limites

A família mais frequente em volume não é necessariamente a mais propensa a movimentar entre os elegíveis. Ponderar as classes é escolha metodológica, não solução automática. Estas métricas não permitem afirmar aceite de oferta ou ganho incremental.

**Condições:** `RETIDO` é o status final, portanto o exercício é retrospectivo; os registros não identificam episódios únicos. Produto presente em M0 não cobre inclusão de família ausente. Consulte `docs/family_imbalance_analysis.md` para interpretações e restrições.